# LayerCAM vs. Transformer Attribution
Run all cells in order in Google Colab. The Chefer ViT needs gradient tracking even for its prediction forward pass because its custom implementation registers hooks.

**Note:** This notebook has not been executed in Colab here; pretrained weights and repository code require an internet-enabled runtime.


In [ ]:
%pip -q install "grad-cam>=1.5,<2" "timm>=0.9,<1.1" opencv-python-headless matplotlib pillow requests
!git clone -q https://github.com/hila-chefer/Transformer-Explainability.git /content/Transformer-Explainability || true
!git -C /content/Transformer-Explainability rev-parse HEAD

In [ ]:
import os, sys, io, requests, numpy as np, torch, matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from pytorch_grad_cam import LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
URLS = [
    'https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg',
]
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
# Chefer's ViT (vit_base_patch16_224) was trained with mean=std=0.5 per channel,
# not the ImageNet mean/std used by ResNet-50, so it gets its own transform.
vit_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def load_image(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    pil = Image.open(io.BytesIO(response.content)).convert('RGB').resize((224,224))
    rgb = np.asarray(pil, dtype=np.float32) / 255.0
    tensor = preprocess(pil).unsqueeze(0).to(DEVICE)
    vit_tensor = vit_preprocess(pil).unsqueeze(0).to(DEVICE)
    return pil, rgb, tensor, vit_tensor

pil, rgb, x, x_vit = load_image(URLS[0])
plt.imshow(pil); plt.axis('off'); plt.title('Input image');

In [ ]:
cnn_weights = ResNet50_Weights.DEFAULT
cnn = models.resnet50(weights=cnn_weights).to(DEVICE).eval()
with torch.no_grad():
    cnn_logits = cnn(x)
cnn_class = int(cnn_logits.argmax(dim=1).item())
cnn_conf = float(cnn_logits.softmax(dim=1)[0, cnn_class])
cnn_label = cnn_weights.meta['categories'][cnn_class]
print('ResNet-50 prediction:', cnn_label, 'confidence:', round(cnn_conf, 4))

# Final convolutional stage, before global pooling
with LayerCAM(model=cnn, target_layers=[cnn.layer4[-1]]) as cam:
    cnn_heatmap = cam(input_tensor=x, targets=[ClassifierOutputTarget(cnn_class)])[0]
cnn_overlay = show_cam_on_image(rgb, cnn_heatmap, use_rgb=True)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(rgb); plt.title('Original'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(cnn_overlay); plt.title('ResNet-50 LayerCAM'); plt.axis('off')
plt.tight_layout()

In [ ]:
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

repo = '/content/Transformer-Explainability'
if repo not in sys.path:
    sys.path.insert(0, repo)

from baselines.ViT.ViT_LRP import vit_base_patch16_224
from baselines.ViT.ViT_explanation_generator import LRP
from pytorch_grad_cam.utils.image import show_cam_on_image

# The custom Chefer ViT registers gradient hooks during its forward pass.
# Do NOT run its forward pass under torch.no_grad() or torch.inference_mode().
vit = vit_base_patch16_224(pretrained=True).to(DEVICE).eval()
explainer = LRP(vit)

# Use a fresh leaf input with gradient tracking enabled.
vit_input = x_vit.detach().clone().to(DEVICE).requires_grad_(True)
with torch.enable_grad():
    vit_logits = vit(vit_input)
vit_class = int(vit_logits.argmax(dim=1).item())
vit_conf = float(vit_logits.softmax(dim=1)[0, vit_class].item())
# Chefer's ViT uses the standard ImageNet-1k class order, so the ResNet-50 label list applies.
vit_label = cnn_weights.meta['categories'][vit_class]
print('ViT prediction:', vit_label, f'(class {vit_class})', 'confidence:', round(vit_conf, 4))

# Attribution also needs gradients. Use a fresh input for the second forward pass.
vit.zero_grad(set_to_none=True)
vit_input = x_vit.detach().clone().to(DEVICE).requires_grad_(True)
with torch.enable_grad():
    relevance = explainer.generate_LRP(
        vit_input, method='transformer_attribution', index=vit_class
    )

# The base patch16_224 model has 14 x 14 image patches.
relevance = relevance.detach().cpu().numpy().reshape(14, 14)
relevance = np.maximum(relevance, 0)
relevance = (relevance - relevance.min()) / (relevance.max() - relevance.min() + 1e-8)
vit_heatmap = cv2.resize(relevance.astype(np.float32), (224, 224), interpolation=cv2.INTER_CUBIC)
vit_heatmap = np.clip(vit_heatmap, 0, 1)
vit_overlay = show_cam_on_image(rgb.astype(np.float32), vit_heatmap, use_rgb=True)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(rgb); plt.title('Original image'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(vit_overlay); plt.title('ViT Transformer Attribution'); plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
os.makedirs('/content/xai_results', exist_ok=True)
fig, axs = plt.subplots(1,3,figsize=(14,4))
for ax, img, title in zip(axs,
    [rgb, cnn_overlay, vit_overlay],
    ['Input', f'LayerCAM: {cnn_label}', f'Transformer Attribution: {vit_label}']):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.tight_layout()
fig.savefig('/content/xai_results/comparison.png', dpi=160, bbox_inches='tight')
plt.show()
Image.fromarray(cnn_overlay).save('/content/xai_results/layercam.png')
Image.fromarray(vit_overlay).save('/content/xai_results/transformer_attribution.png')
print('Saved outputs to /content/xai_results/')

## 6. Compare three more images (Colab upload)
Run this section **after** the five existing code cells. Select **three image files** (JPG or PNG) from your computer when prompted. The models are reused; they are **not downloaded again**. Each image produces a three-panel figure and a row in a CSV file. Predictions are from different model architectures, so compare explanations qualitatively rather than claiming one method is more accurate based on the heatmaps alone.

**Privacy:** Only upload images you have permission to use. This cell stores copies of outputs in the temporary Colab runtime at `/content/xai_results/multi_image/`. Download the ZIP before the runtime resets.


In [ ]:
# Test three more images with the already-loaded CNN and ViT.
# This cell is intended for Google Colab and runs after the existing cells.
from google.colab import files
from pathlib import Path
import csv, re, zipfile

print('Choose 3 JPG/PNG images from your computer.')
uploaded = files.upload()
image_files = [(name, data) for name, data in uploaded.items()
               if name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
if len(image_files) != 3:
    raise ValueError(f'Please upload exactly 3 images; found {len(image_files)}. Re-run this cell to try again.')

out_dir = Path('/content/xai_results/multi_image')
out_dir.mkdir(parents=True, exist_ok=True)
results = []

# Both models use the standard ImageNet-1k class order, so one label list serves both.
for number, (filename, image_bytes) in enumerate(image_files, start=1):
    print(f'Processing {number}/3: {filename}')
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB').resize((224, 224))
    image_rgb = np.asarray(image, dtype=np.float32) / 255.0
    image_x = preprocess(image).unsqueeze(0).to(DEVICE)          # ResNet-50 input
    image_x_vit = vit_preprocess(image).unsqueeze(0).to(DEVICE)  # ViT input (mean=std=0.5)

    # CNN prediction and LayerCAM
    with torch.no_grad():
        cnn_output = cnn(image_x)
    c_idx = int(cnn_output.argmax(dim=1).item())
    c_prob = float(cnn_output.softmax(dim=1)[0, c_idx].item())
    c_label = cnn_weights.meta['categories'][c_idx]
    with LayerCAM(model=cnn, target_layers=[cnn.layer4[-1]]) as cam:
        c_heat = cam(input_tensor=image_x, targets=[ClassifierOutputTarget(c_idx)])[0]
    c_overlay = show_cam_on_image(image_rgb, c_heat, use_rgb=True)

    # Chefer custom ViT requires gradients even for prediction forwards.
    v_input = image_x_vit.detach().clone().requires_grad_(True)
    with torch.enable_grad():
        v_output = vit(v_input)
    v_idx = int(v_output.argmax(dim=1).item())
    v_prob = float(v_output.softmax(dim=1)[0, v_idx].item())
    v_label = cnn_weights.meta['categories'][v_idx]

    vit.zero_grad(set_to_none=True)
    v_input = image_x_vit.detach().clone().requires_grad_(True)
    with torch.enable_grad():
        rel = explainer.generate_LRP(v_input, method='transformer_attribution', index=v_idx)
    rel = rel.detach().cpu().numpy().reshape(14, 14)
    rel = np.maximum(rel, 0)
    rel = (rel - rel.min()) / (rel.max() - rel.min() + 1e-8)
    v_heat = cv2.resize(rel.astype(np.float32), (224, 224), interpolation=cv2.INTER_CUBIC)
    v_overlay = show_cam_on_image(image_rgb, np.clip(v_heat, 0, 1), use_rgb=True)

    safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', Path(filename).stem)[:50]
    prefix = f'{number:02d}_{safe_name}'
    image.save(out_dir / f'{prefix}_original.png')
    Image.fromarray(c_overlay).save(out_dir / f'{prefix}_layercam.png')
    Image.fromarray(v_overlay).save(out_dir / f'{prefix}_transformer_attribution.png')

    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, img, heading in zip(axs,
        [image_rgb, c_overlay, v_overlay],
        ['Input', f'LayerCAM: {c_label} ({c_prob:.1%})',
         f'Transformer Attribution: {v_label} ({v_prob:.1%})']):
        ax.imshow(img)
        ax.set_title(heading, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    fig.savefig(out_dir / f'{prefix}_comparison.png', dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    results.append({'image': filename, 'resnet50_class_id': c_idx,
                    'resnet50_label': c_label, 'resnet50_confidence': round(c_prob, 4),
                    'vit_class_id': v_idx, 'vit_label': v_label, 'vit_confidence': round(v_prob, 4),
                    'your_observations': ''})

csv_path = out_dir / 'prediction_results.csv'
with csv_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

zip_path = Path('/content/xai_results/multi_image_results.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in out_dir.iterdir():
        if item.is_file():
            archive.write(item, arcname=item.name)

print('Saved comparison figures, overlays, and CSV:', out_dir)
print('ZIP ready:', zip_path)
print('Fill in the your_observations column yourself after examining the figures.')
files.download(str(zip_path))


## 7. Your analysis notes (complete yourself)
For each image, note: (1) whether the two models predict the same class, (2) where LayerCAM concentrates its activation, (3) where Transformer Attribution concentrates its relevance, and (4) any surprising or ambiguous regions. Describe what you actually see; do not treat heatmap intensity as a direct measurement of explanation quality. Your course requires that the four-page report be written from your own analysis and observations.
